In [2]:
import json
from src.adapters import ProviderAdapter, AnthropicAdapter, OpenAIAdapter, DeepseekCoTAdapter, OpenAICompatibleAdapter
import src.utils as utils
from src.models import ARCTaskOutput, ARCPair
from src.prompts.prompt_manager import convert_task_pairs_to_prompt
from typing import List, Any, Optional
# load_dotenv
import os
from dotenv import load_dotenv
load_dotenv()


True

In [3]:
# generation_model_name = "agentica-org/DeepScaleR-1.5B-Preview"
# generation_base_url = "http://100.95.78.36:8000/v1"
# generation_model_name = "deepseek-reasoner"
# generation_base_url = "https://api.deepseek.com"
# extraction_model_name = "deepseek-reasoner"
# extraction_base_url = "https://api.deepseek.com"
# parse_cot = False
# provider =DeepseekCoTAdapter(generation_model_name, generation_base_url, extraction_model_name, extraction_base_url, parse_cot)

In [5]:
# Local unsloth/Qwen2.5-7B-Instruct
model_name = "unsloth/Qwen2.5-Math-7B"
base_url = "http://localhost:8000/v1"
provider =OpenAICompatibleAdapter(model_name, base_url)

In [6]:
task_id = "7c008303"
data_dir = "data/arc-agi/data/training"
train_pairs = utils.get_train_pairs_from_task(data_dir, task_id)
test_input = utils.get_test_input_from_task(data_dir, task_id)
test_input_pair = test_input[0]
print(test_input)

[ARCPair(input=[[0, 0, 0, 3, 0, 0, 8, 0, 0], [3, 3, 0, 3, 0, 3, 8, 0, 0], [0, 3, 0, 3, 0, 3, 8, 0, 0], [0, 3, 3, 3, 0, 0, 8, 0, 0], [0, 3, 0, 0, 0, 3, 8, 0, 0], [0, 0, 3, 0, 0, 0, 8, 0, 0], [8, 8, 8, 8, 8, 8, 8, 8, 8], [0, 0, 0, 0, 0, 0, 8, 2, 1], [0, 0, 0, 0, 0, 0, 8, 4, 7]], output=None)]


## Generate Prompts

In [7]:
prompt = convert_task_pairs_to_prompt(train_pairs, test_input_pair)
print(prompt)

You are participating in a puzzle solving competition. You are an expert at solving puzzles.

Below is a list of input and output pairs with a pattern. Your goal is to identify the pattern or transformation in the training examples that maps the input to the output, then apply that pattern to the test input to give a final output.

Respond in the format of the training output examples

--Training Examples--
--Example 0-- 

 INPUT: 

[2, 4, 8, 0, 0, 0, 0, 0, 0]
[1, 6, 8, 0, 0, 0, 0, 0, 0]
[8, 8, 8, 8, 8, 8, 8, 8, 8]
[0, 0, 8, 0, 3, 0, 0, 3, 0]
[0, 0, 8, 3, 3, 3, 3, 3, 3]
[0, 0, 8, 0, 3, 0, 0, 3, 0]
[0, 0, 8, 0, 3, 0, 0, 3, 0]
[0, 0, 8, 3, 3, 3, 3, 3, 3]
[0, 0, 8, 0, 3, 0, 0, 3, 0]


OUTPUT: 

[0, 2, 0, 0, 4, 0]
[2, 2, 2, 4, 4, 4]
[0, 2, 0, 0, 4, 0]
[0, 1, 0, 0, 6, 0]
[1, 1, 1, 6, 6, 6]
[0, 1, 0, 0, 6, 0]


--Example 1-- 

 INPUT: 

[0, 0, 0, 0, 0, 0, 8, 1, 2]
[0, 0, 0, 0, 0, 0, 8, 4, 1]
[8, 8, 8, 8, 8, 8, 8, 8, 8]
[0, 0, 3, 3, 0, 3, 8, 0, 0]
[3, 3, 0, 0, 0, 0, 8, 0, 0]
[3, 3, 0, 3, 0, 3

## Test Single Response from LLM

In [8]:
response = provider.make_prediction(prompt)
print(response)

BadRequestError: Error code: 400 - {'object': 'error', 'message': 'As of transformers v4.44, default chat template is no longer allowed, so you must provide a chat template if the tokenizer does not define one.', 'type': 'BadRequestError', 'param': None, 'code': 400}

## Test generating response and extracting JSON

In [7]:
from main import ARCTester
save_submission_dir = "submissions/pipeline"
overwrite_submission = False
print_submission = True
num_attempts = 1
retry_attempts = 0
print_logs = True
arc_tester = ARCTester("openai_compatible", model_name, save_submission_dir, overwrite_submission, print_submission, num_attempts, retry_attempts, print_logs,base_url=base_url)

In [8]:
task_output = arc_tester.predict_task_output(train_pairs, test_input_pair)

Making prediction for task


In [9]:
print(task_output)

[0, 0, 0, 4, 0, 0]
[4, 4, 0, 4, 0, 4]
[0, 4, 0, 4, 0, 4]
[0, 4, 4, 4, 0, 0]
[0, 4, 0, 0, 0, 4]
[0, 0, 4, 0, 0, 0]
[0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0]


In [10]:
json_response = arc_tester.parse_and_validate_json(task_output)
print(json_response)

[[0, 0, 0, 4, 0, 0], [4, 4, 0, 4, 0, 4], [0, 4, 0, 4, 0, 4], [0, 4, 4, 4, 0, 0], [0, 4, 0, 0, 0, 4], [0, 0, 4, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]]
